# 02 — Train Baseline (M4)

**GPU training entry point** for the EfficientNet-B0 transfer-learning baseline.

This notebook is a *thin driver*: all real logic lives in `src/maize_detection/`
(`model.py`, `train.py`, `data.py`). Here we only orchestrate the run on a GPU
runtime (Colab / Kaggle), then save the best checkpoint.

**Two-phase transfer learning:**
- **Phase A** — freeze the ImageNet backbone, train only the new 4-class head.
- **Phase B** — unfreeze everything, fine-tune end-to-end at a lower LR.

The single best-by-validation-loss checkpoint is written to
`outputs/checkpoints/best.pt`.

> ⚠️ **Data is not in git.** The PlantVillage images and the split manifest are
> `.gitignore`d, so they must be downloaded *inside this runtime* (next section).
> Push your latest code to GitHub before running on Colab so the clone is current.

## 1. Environment setup (Colab)

Clones the repo into the Colab runtime and installs the few pure-Python deps
(torch + torchvision are preinstalled on Colab with CUDA). On a local machine
this block is skipped and we just use the current working directory.

In [ ]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/CaiZhengTech/MaizeDetection.git"
    BRANCH = "dev"  # the branch you pushed your M4 code to
    if not Path("MaizeDetection").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    %cd MaizeDetection
    # torch/torchvision already on Colab; install the rest quietly
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn", "pyyaml", "tqdm", "pillow", "numpy"], check=True)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("In Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

## 2. Get data + build the leaf-grouped splits

Both steps are **idempotent** — they only run if their output is missing.

- `download_plantvillage.py` pulls the corn images (~2 GB into the runtime) and
  writes `plantvillage_manifest.csv`.
- `python -m src.maize_detection.data` builds the leaf-grouped 70/15/15 split
  manifest (`plantvillage_splits.csv`) — the same anti-leakage splitter the M3
  test guards.

In [ ]:
manifest = PROJECT_ROOT / "data/processed/plantvillage_manifest.csv"
splits   = PROJECT_ROOT / "data/processed/plantvillage_splits.csv"

if not manifest.exists():
    subprocess.run([sys.executable, "scripts/download_plantvillage.py"], check=True)
if not splits.exists():
    subprocess.run([sys.executable, "-m", "src.maize_detection.data"], check=True)

print("manifest present:", manifest.exists())
print("splits present:  ", splits.exists())

## 3. Confirm the GPU

Training is intended to run on a GPU. If this prints `cpu`, switch the runtime:
**Runtime → Change runtime type → Hardware accelerator → GPU**, then re-run.

In [ ]:
import torch
from src.maize_detection.utils import get_device

device = get_device()
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — training will be slow. Switch the runtime to GPU.")

## 4. Train

`run_training(cfg)` runs both phases using the epoch counts and learning rates in
`configs/baseline.yaml` (Phase A: 5 epochs @ 1e-3; Phase B: 15 epochs @ 1e-4).

**Tip:** do a fast pipeline check first by capping the work, then run the real
thing:

```python
run_training(cfg, max_batches=2, epochs_a=1, epochs_b=1)   # smoke run (~seconds)
```

In [ ]:
from src.maize_detection.config import load_config
from src.maize_detection.train import run_training

cfg = load_config()
print("seed:", cfg["seed"])
print("Phase A:", cfg["training"]["phase_a"], "| Phase B:", cfg["training"]["phase_b"])

# Full run (uses the config's epoch counts). Pass max_batches=... for a smoke run.
result = run_training(cfg)

print("\nBest val loss:", round(result["best_val_loss"], 4))
print("Checkpoint:   ", result["checkpoint"])

## 5. Save the checkpoint

On Colab the runtime is ephemeral — download `best.pt` so you can run **local CPU
inference** and the M5/M6 evaluations with it. Place it at
`outputs/checkpoints/best.pt` in your local repo.

In [ ]:
ckpt = result["checkpoint"]
print("Saved:", ckpt.exists(), "| size MB:", round(ckpt.stat().st_size / 1e6, 1))

if IN_COLAB:
    from google.colab import files
    files.download(str(ckpt))

## Next: M5 — In-domain evaluation

With `best.pt` in `outputs/checkpoints/`, the next milestone evaluates it on the
held-out **controlled** test split: accuracy, per-class precision/recall/F1,
macro-F1, confusion matrix, and the per-class **false-negative rate**.